# Ray Train In Detail

This notebook provides a deep dive into the internals of Ray Train.

- Execution of `Trainer.fit()`
- Checkpointing
- Ray Data integration

<div class="alert alert-block alert-info" style="margin-top: 20px">
<li>Overview of Ray Train</li>
<li>Train Execution Deep Dive</li>
<li>Checkpointing Deep Dive</li>
<li>Ray Data Integration Internals</li>
</div>

## Overview of Ray Train

Here is a diagram showing the high-level architecture of Ray Train:

<img src="https://docs.ray.io/en/latest/_images/overview.png" alt="Trainer.fit() execution" width="800">

It involves the following components:
- **Training function**: A Python function that contains your model training logic.
- **Worker**: A process that runs the training function.
- **Scaling configuration**: A configuration of the number of workers and compute resources (for example, CPUs or GPUs).
- **Trainer**: A Python class that ties together the training function, workers, and scaling configuration to execute a distributed training job.



## Trainer Execution Deep Dive

Below is a diagram showing the execution of `Trainer.fit()` in the current implementation of Ray Train (V1):

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/pinterest/train_tune_dependency.png" alt="Trainer.fit() execution" width="800">


It turns out that the execution of `Trainer.fit()` is tightly coupled with the execution of `Tune.run()`. 

This internal coupling is not directly exposed to the user in Ray Train via the high-level docs but it leaks into the implementation in a few places (checkpointing, logs, and metrics).



### Execution of Trainer.fit() in V2

Below is a diagram showing the proposal for a Ray Train (V2) with a decoupled implementation of Train and Tune.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/pinterest/train_tune_decoupled.png" alt="Trainer.fit() execution" width="800">


For an in-depth look at the changes in the API and the decoupling of Train and Tune, please refer to the [revamp proposal](https://github.com/ray-project/enhancements/blob/10df195ea7750d1f4571c483e9ca4de5afada0ea/reps/2024-10-18-train-tune-api-revamp/2024-10-18-train-tune-api-revamp.md).

## Execution of Trainer.fit() in more detail

Let's focus on how a Trainer driver launches a training job and tracks its progress. 

Here is a diagram detailing the execution of `Trainer.fit()`:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/pinterest/training_internals_v2.png" alt="Trainer.fit() internals" width="900">



Here is the same diagram adapted for the case of running a `TorchTrainer`:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/pinterest/torch_training_internals_v2.png" alt="TorchTrainer.fit() internals" width="900">



Note that `Ray Train` is not re-implementing complex distributed training logic. It is instead using the existing distributed training logic in the training frameworks like pytorch.

What Ray Train offers is a high-level interface for users to:
- create training worker processes on a cluster with multiple GPUs
- enable fault-tolerance capabilities
    - checkpointing and resuming training jobs
- get observability into the training job (logs, metrics, stack traces, profiling)
- leverage Ray Data integration to allow for:
    - online preprocessing and independent scaling of data and training across a heterogeneous cluster

## Ray Train's Checkpointing In Detail

### Checkpointing

Here are the steps that take place when a user reports a checkpoint directory:

- Step 1: User code thread reports a local checkpoint directory to a persistent store.
- Step 2: Listener thread pushes the checkpoint reference to the train controller.
- Step 3: Train controller pushes the checkpoint reference to the Tune controller.
- Step 4: Tune controller updates the latest checkpoint reference.

Here is a diagram of the above steps:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/pinterest/checkpointing_v2.png" alt="Checkpoint lifecycle" width="1000">


- A `train.Checkpoint` is just a wrapper around a directory path.
  - The contents are completely up to the user:
    - User can choose their own serialization format.
    - Easy to use familiar checkpoint utilities provided by training frameworks, such as `torch.save`, `pl.Trainer.save_checkpoint`, `Accelerate’s accelerator.save_model`, `tf.keras.Model.save`, etc.
  - Report the checkpoint to Ray Train using `ray.train.report(metrics, checkpoint=...)`
    - The metrics reported alongside the checkpoint are used to keep track of the best-performing checkpoints.
    - `ray.train.report` will upload the checkpoint to persistent storage if configured. 

Here is a diagram showing the lifecycle of a Checkpoint, from being saved locally to disk to being uploaded to persistent storage via `ray.train.report`.

<img src="https://docs.ray.io/en/latest/_images/checkpoint_lifecycle.png" alt="Checkpoint lifecycle" width="800">

### New Features in V2

The new version of Ray Train introduces several key improvements:

- **Decoupled Implementations**:
  - Expands checkpointing features, especially for multi-worker distributed checkpointing.
  - Avoids the need to pass checkpoints through multiple layers, allowing for more Train-specific functionalities.

- **Improved Checkpoint Path Customization**:
  - Provides capability to specify deterministic checkpoint paths to simplify post-training access and management.

- **Avoid periodic storage commits**:
  - Ensures a more reliable restoration process.


### Distributed Checkpointing with FSDP

When doing model-parallel training, Ray Train checkpointing provides an easy way to upload model shards from each worker in parallel, without needing to gather the full model to a single node.

In model parallel training strategies where each worker only has a shard of the full-model, you can save and report checkpoint shards in parallel from each worker.

<img src="https://docs.ray.io/en/latest/_images/persistent_storage_checkpoint.png" alt="Distributed checkpointing" width="500">

Each worker uploads its own checkpoint shard to persistent storage independently.

Distributed checkpointing is the best practice for saving checkpoints when doing model-parallel training (e.g., DeepSpeed, FSDP, Megatron-LM).

There are two major benefits:

- It is faster, resulting in less idle time. Faster checkpointing incentivizes more frequent checkpointing!
    - Each worker can upload its checkpoint shard in parallel, maximizing the network bandwidth of the cluster. Instead of a single node uploading the full model of size M, the cluster distributes the load across N nodes, each uploading a shard of size M / N.
- Distributed checkpointing avoids needing to gather the full model onto a single worker’s CPU memory.
    - This gather operation puts a large CPU memory requirement on the worker that performs checkpointing and is a common source of OOM errors.

The script `fsdp_ckpt.py` demonstrates how to perform distributed checkpointing with FSDP.


### In-epoch checkpoint support

Ray Train supports performing checkpointing at any point in the training loop. i.e. a user can call `ray.train.report` at any point in the training worker loop.


Challenges:
- Resuming from the same batch of data within an epoch is not yet supported.
- To avoid resuming training and running over data that has already been trained, the workaround is to shuffle data every epoch:
    - i.e. making use of `read_parquet(shuffle='files')` and `iter_torch_batches(local_shuffle_buffer_size=N * batch_size)`


i.e.

```python
ds = ray.data.read_parquet("s3://my-bucket/my-dataset", shuffle='files')

def train_loop_per_worker(config):
    # ...
    batch_iterator = ds_shard.iter_torch_batches(local_shuffle_buffer_size=N * batch_size)
    for batch_idx, batch in enumerate(batch_iterator):
        # ...
        ray.train.report(metrics, checkpoint=...)

trainer = Trainer(
    train_loop_per_worker=train_loop_per_worker,
    scaling_config=ScalingConfig(num_workers=4, use_gpu=True),
    datasets={"train": train_dataset, "val": val_dataset},
)

trainer.fit()
```


## Ray Train + Ray Data Integration

A user passes `datasets` to a `Trainer` object .

The `datasets` specifies the Ray Dataset objects to iterate on when running the distributed training

Here is some sample pseudocode showing how a user might use Ray Data with Ray Train:

```python
train_dataset = ray.data.read_parquet("s3://my-bucket/my-dataset").map_batches(preprocess_fn)
val_dataset = ray.data.read_parquet("s3://my-bucket/my-dataset").map_batches(preprocess_fn)

trainer = Trainer(
    train_loop_per_worker=train_loop_per_worker,
    scaling_config=ScalingConfig(num_workers=4, use_gpu=True),
    datasets={"train": train_dataset, "val": val_dataset},
)

trainer.fit()
```


### DataConfig and Dataset Splitting

Ray Train uses a default `DataConfig` to configure how the datasets are split across the training workers.

i.e. by default, here is the code that is executed:

```python
trainer = Trainer(
    train_loop_per_worker=train_loop_per_worker,
    scaling_config=ScalingConfig(num_workers=4, use_gpu=True),
    datasets={"train": train_dataset, "val": val_dataset},
    data_config=DataConfig(datasets_to_split="all"),
)
```

The Trainer driver will run `DataConfig.configure` to produce the dataset shards that are used to initialize the training session on each worker - i.e. the `Trainer` driver will coordinate the splitting of the datasets.

To verify this, take a look at the `BackendExecutor.start_training` method:

In [ ]:
from ray.train._internal.backend_executor import BackendExecutor

%psource BackendExecutor.start_training

The `DataConfig.configure` method specifies how to split the datasets across the training workers

Here is the default implementation for `DataConfig.configure`:

In [ ]:
%psource DataConfig.configure

For each dataset:
- Update its execution options to exclude using the resources reserved for the training workers
- If it is specified as one of the `datasets_to_split` then `ds.streaming_split` is used to equally split the dataset across all the training workers.
- Otherwise, `ds.iterator` is used to provide the full dataset iterator to each training worker.

### Streaming Split Internals

`Dataset.streaming_split` will generate `n` `StreamingSplitDataIterator` objects, where `n` is the number of training workers.

For each dataset, `SplitCoordinator` actors are created to coordinate the splitting of the dataset across the `StreamingSplitDataIterator` objects.

The `SplitCoordinator` actor is responsible for:
- Pulling block references from the executed stream 
- Dividing those blocks among `n` output iterators
- Returning the blocks to the `StreamingSplitDataIterator` objects


Additionally, the `SplitCoordinator` actor:
- Adds a barrier at the start of each iteration to ensure all iterators are ready and synchronized to start the iteration
- Ensures that the iterators are repeatable, so a new epoch can be started.


Here is what `Dataset.streaming_split` does under the hood:


In [ ]:
from ray.data.dataset import Dataset

%psource Dataset.streaming_split

where the `StreamSplitDataIterator.create` performs the following steps:

In [ ]:
from ray.data._internal.iterator.stream_split_iterator import StreamSplitDataIterator

%psource StreamSplitDataIterator.create

In [ ]:
%psource StreamSplitDataIterator._to_block_iterator

Two main `SplitCoordinator` methods are `start_epoch` and `get`

In [ ]:
from ray.data._internal.iterator.stream_split_iterator import SplitCoordinator

%psource SplitCoordinator.start_epoch

In [ ]:
%psource SplitCoordinator._barrier

In [ ]:
%psource SplitCoordinator.get

### DatasetIterator.iter_batches in detail

Whether an iterator is generated by `streaming_split` or `iterator`, the `iter_batches` method is the same.

The `iter_batches` method efficiently creates formatted data batches from an iterator of block object references and metadata. It leverages both pipeline and data parallelism for optimal performance.

- **Batch Creation**: Slices, unions, shuffles, prefetches, and formats blocks into specified batch sizes.
- **Parallelism**: Utilizes pipeline and data parallelism to process multiple batches concurrently.
- **Prefetching**: Optimizes data retrieval by prefetching a set number of batches.
- **Thread Management**: Uses a threadpool to format and collate batches, ensuring ordered output.

**Steps Involved**:
1. Prefetch block references.
2. Resolve references with `ray.get()`.
3. Slice and shuffle to form batches.
4. Format and collate in parallel threads.
5. Finalize and output batches sequentially.

This method is designed to handle large datasets efficiently using Ray's distributed computing features.

In [ ]:
from ray.data.iterator import DatasetIterator

%psource DatasetIterator.iter_batches

In [ ]:
from ray.data._internal.block_batching.iter_batches import iter_batches

%psource iter_batches